# Ch 14. LangChain 실전 + 멀티모달 RAG

이 노트북은 [AI Assistant Engineering - Part 3, Ch 14](https://desty.github.io/study-ai-assistant-engineering/part3/14-langchain-multimodal/) 의 실습용 보조 자료입니다.

## 주제
- **LangChain** 의 RAG 컴포넌트 (Loader · Splitter · Embeddings · VectorStore · Retriever · Chain)
- **LCEL** (LangChain Expression Language) 로 파이프라인을 한 줄로 조립
- **대화형 RAG** — 이전 대화를 고려한 검색·응답
- **멀티모달 RAG** — PDF 레이아웃·이미지까지 검색 대상
- LangChain 버전 불안정성 · OCR 품질 · 테이블 파싱 — 실전 3대 함정

In [ ]:
!pip install -q langchain langchain-community langchain-anthropic langchain-openai langchain-chroma unstructured

In [ ]:
import os
from getpass import getpass

for k in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY']:
    if not os.getenv(k):
        os.environ[k] = getpass(f'{k}: ')

## 1. 개념 — 프레임워크를 왜 지금?

Ch 11~13 까지 우리는 **수동** 으로 RAG를 조립했습니다.

**LangChain** 은 이걸 **레고 블록** 처럼 묶는 프레임워크.

> LangChain 은 **생산성 도구** 이지 **이해의 지름길** 이 아닙니다.

## 2. LangChain RAG 컴포넌트

| 영역 | 컴포넌트 | 우리가 Ch 11에서 직접 쓴 것 |
|---|---|---|
| **Indexing** | `DocumentLoader` | PyPDF · `Path.read_text` |
| | `TextSplitter` | 간단 분할 (또는 `RecursiveCharacterTextSplitter`) |
| | `Embeddings` | `openai.embeddings.create(...)` |
| | `VectorStore` | Chroma 직접 클라이언트 |
| **Query** | `Retriever` | `col.query(...)` |
| | `PromptTemplate` | f-string |
| | `ChatModel` | `anthropic.Anthropic().messages.create` |
| | `OutputParser` | `response.content[0].text` |

## 4. 최소 예제 — Ch 11의 mini_rag 를 LangChain 으로

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1) 벡터스토어
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(
    collection_name="policies",
    embedding_function=embeddings,
    persist_directory="./lc_db_ch14"
)

# 2) 문서 추가
docs_text = [
    "환불은 7일 이내, 팀장 승인 필요.",
    "배송은 2~3일, 도서산간 +2일."
]
from langchain_core.documents import Document
docs = [Document(page_content=text) for text in docs_text]
vectorstore.add_documents(docs)

# 3) Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 4) 프롬프트 + 모델 + 파서 = Chain (LCEL)
prompt = ChatPromptTemplate.from_template("""다음 문서만 근거로 답하세요. 문서에 없으면 "문서에 없습니다".

{context}

질문: {question}""")

model = ChatAnthropic(model="claude-haiku-4-5", max_tokens=512)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

answer = chain.invoke("환불하고 싶은데")
print("=== LangChain RAG 답변 ===")
print(answer)

## 5. 실전 튜토리얼

### 5.1 LCEL — Chain 조립

In [ ]:
from langchain_core.runnables import RunnableParallel

# 병렬 체인: 같은 입력을 여러 체인에 보내고 결과 합침
parallel_chain = RunnableParallel(
    answer = chain,
    sources = retriever,  # 원본 문서도 반환
)

result = parallel_chain.invoke("환불 기한?")
print("답변:", result["answer"])
print("출처:", result["sources"])

LCEL 의 3가지 이득:

1. **비동기 자동 지원** (`ainvoke`, `astream`)
2. **스트리밍 내장** (`.stream(...)`)
3. **병렬성** (RunnableParallel)

### 5.2 대화형 RAG — History 반영

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain.chains import create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# history 고려해 쿼리를 재작성하는 retriever
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "이전 대화를 고려해 현재 질문을 자립형 질문으로 재작성하세요."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(
    model, retriever, contextualize_prompt
)

# 실제 답변 생성
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "아래 문서만 근거로 답하세요.\n\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
qa_chain = create_stuff_documents_chain(model, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

# 사용
history = []
r1 = rag_chain.invoke({"input": "환불하려고", "chat_history": history})
print("Q1: 환불하려고")
print(f"A1: {r1['answer'][:200]}...\n")

history.append(("human", "환불하려고"))
history.append(("assistant", r1["answer"]))

r2 = rag_chain.invoke({"input": "승인 누가 해?", "chat_history": history})
print("Q2: 승인 누가 해?")
print(f"A2: {r2['answer'][:200]}...")

### 5.3 멀티모달 RAG — 텍스트 + 이미지

PDF 에는 **표·다이어그램·스크린샷** 이 있습니다. 텍스트만 추출하면 의미 손실.

**두 가지 접근**:
1. **이미지를 직접 임베딩** — CLIP · Voyage `voyage-multimodal-3`
2. **이미지를 LLM 으로 설명** — Claude/GPT-4o 로 설명 텍스트 생성 후 텍스트 임베딩

**방법 2가 실전 권장** — 한국어·도메인 특화 검색에서 정확도↑, 관리 쉬움.

In [ ]:
# PDF 멀티모달 처리 예시 (실제 PDF 파일 필요)
# import fitz  # PyMuPDF
# from anthropic import Anthropic
# import base64

# anthropic = Anthropic()
# pdf = fitz.open("report.pdf")

# for page_num in range(len(pdf)):
#     page = pdf[page_num]
#     # 텍스트 추출
#     text = page.get_text()
#     if text.strip():
#         index_text_chunk(text, source=f"report.pdf#p={page_num}")

#     # 이미지 추출 → Claude Vision 으로 설명
#     pix = page.get_pixmap(dpi=150)
#     img_bytes = pix.tobytes("png")
#     img_b64 = base64.b64encode(img_bytes).decode()

#     r = anthropic.messages.create(
#         model="claude-haiku-4-5", max_tokens=512,
#         messages=[{
#             "role": "user",
#             "content": [
#                 {"type": "image", "source": {
#                     "type": "base64", "media_type": "image/png", "data": img_b64
#                 }},
#                 {"type": "text", "text": "이 페이지의 표·다이어그램·차트를 한국어로 설명하세요. 숫자는 정확히."},
#             ],
#         }],
#     )
#     caption = r.content[0].text
#     index_text_chunk(caption, source=f"report.pdf#p={page_num}#visual")

print("멀티모달 RAG 코드는 PDF 파일이 필요합니다. 주석 제거 후 실제 PDF로 테스트하세요.")

### 5.4 PDF 레이아웃 심화

표·제목·바디 텍스트를 **구분해서 파싱**:

```python
# 옵션 1: unstructured (고품질, 유료 API 옵션 있음)
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="report.pdf",
    strategy="hi_res",                 # 느림·정확
    infer_table_structure=True,        # 표 구조 보존
    extract_images_in_pdf=True,        # 이미지도 꺼냄
)

for el in elements:
    if el.category == "Table":
        # 표는 마크다운 등으로 특별 취급
        index_text_chunk(el.metadata.text_as_html, source=..., doc_type="table")
    elif el.category == "Title":
        # 섹션 제목으로 청크 경계
        ...
    else:
        index_text_chunk(str(el), source=...)
```

다른 옵션:
- **docling** (IBM, 2024) — 빠르고 정확, 오픈소스
- **PyMuPDF (fitz)** — 가볍고 빠름, 레이아웃 인식은 약함
- **Amazon Textract · Azure Document Intelligence** — 유료 · 한국어 가능

## 6. 자주 깨지는 포인트

### 실수 1. LangChain 버전 불안정성
LangChain 은 API 가 **자주 바뀝니다**. 몇 달 전 튜토리얼 코드가 import 부터 깨지는 경우 흔함.

**대응**: `requirements.txt` 에 **정확한 버전 pin**. 최신은 `langchain-community` · `langchain-{vendor}` 로 분리된 구조.

### 실수 2. OCR 품질이 검색 품질의 상한
한국어 PDF · 스캔 문서 · 오래된 PDF 는 OCR 오류가 많음.

**대응**: (a) 문서 품질 검증 (임의 샘플 수동 확인), (b) 중요 문서는 **원본 텍스트 확보** (Markdown 변환), (c) OCR 엔진 비교.

### 실수 3. 표 파싱 실패
`pypdf` 는 표를 **행 무시하고 텍스트만** 뽑음 → "A B C D 1 2 3 4" 처럼 뒤섞인 텍스트.

**대응**: `unstructured` · `docling` 의 `infer_table_structure=True` · 표는 **별도 chunk·doc_type** 로 관리.

### 실수 4. 멀티모달 비용 폭주
모든 페이지를 Vision LLM 에 돌리면 **페이지당 $0.01~0.05**. 1000페이지면 $50.

**대응**: (a) 이미지가 있는 페이지만 선별, (b) Haiku 같은 **저가 Vision 모델**, (c) **증분 처리** (새 문서만).

### 실수 5. LangChain 으로 마법 기대
컴포넌트가 많아 "대충 엮으면 된다" 생각. 내부 동작 모르면 **버그·성능 문제 진단 불가**.

**대응**: Ch 11~13 의 수동 구현 경험을 **반드시** 먼저. LangSmith 로 tracing 걸어 내부 흐름 관찰.

## 8. 확인 문제

1. 위 LangChain 코드를 실제로 돌리고, 내 문서 5~10개로 간단 RAG 봇 만들기
2. LCEL로 **retriever → prompt → model → parser** 를 한 줄로 조립해 동작 확인
3. 대화형 RAG에서 history가 없을 때와 있을 때 검색·응답이 달라지는 예 찾기
4. PDF 파일이 있다면 §5.3 멀티모달 코드를 실행 → 이미지 캡션이 정확하게 생성되는지 확인
5. LangChain 버전 호환성 테스트 — 현재 환경의 `langchain*` 패키지 버전 기록하고 6개월 뒤 다시 실행해 API 변경 경험

---

## Part 3 완주

Ch 9~14 를 통해 RAG 의 **전체 흐름** 을 배웠습니다:

- **Ch 9**: RAG 가 왜 필요한지, 언제 쓸지
- **Ch 10**: 임베딩 · 벡터 검색의 수학
- **Ch 11**: end-to-end 파이프라인 구축
- **Ch 12**: 검색 실패의 원인과 개선 기법
- **Ch 13**: 기본 RAG 에서 막혀서 나가는 Advanced 기법
- **Ch 14**: 프로덕션 프레임워크 + 멀티모달 까지

다음은 **Part 4 - 평가와 추론** 으로 나아갑니다. RAG 시스템이 제대로 작동하는지 어떻게 검증할지.